# SBN — Statistical Modelling of Design Structures

**Dataset** : *(see Shared Setup — place your CSV in the `data/` directory)*  
**GA parameters** : population = 10, generations = 90  
**Design space** : {0,1}⁶ = 64 architectures  
**Fitness** : `differential_resistance` / `linear_resistance` / `algebraic_resistance`

| Constraint | Symbol | Definition |
|---|---|---|
| Stratification | S | Alternating nonlinear / linear layers |
| Acyclicity     | A | Strict DAG — no feedback cycles |
| Regularity     | R | Equal source-to-sink path lengths |
| Interleaving   | I | Cross-block dependencies (B0 ↔ B1) |
| Homogeneity    | H | Identical operations within each layer |
| Locality       | L | Bounded connection distance |

---

## Table of Contents

- **Step 0** — Completeness check & multi-level discretisation  
- **Step 1** — Hypercube edge analysis  
- **Step 2** — Epistasis matrix  
- **Step 3** — FCA lattice for minimal rules  
- **Step 4** — Post-lattice validation  
- **Step 5** — Regression model & importance table  

> Single run per architecture — results are structural tendencies, not statistical certainties.

---
## Shared Setup

In [ ]:
import os as _os, pandas as pd, numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from matplotlib.colors import TwoSlopeNorm
from IPython.display import display

# ── CSV file — USER ACTION REQUIRED ─────────────────────────────────────────
# Place your ga_results_*.csv file in the data/ subdirectory next to this
# notebook, then update the filename below.
# Example filenames:
#   ga_results_algebraic_pop10_gen90.csv
#   ga_results_linear_pop10_gen90.csv
#   ga_results_differential_pop10_gen90.csv
_fname   = 'ga_results_algebraic_pop10_gen90.csv'  # <── change this
CSV_PATH = _os.path.join('data', _fname)
if not _os.path.exists(CSV_PATH):
    # fallback: search current directory
    CSV_PATH = next((p for p in [
        _fname,
        _os.path.join(_os.getcwd(), _fname),
    ] if _os.path.exists(p)), None)
if CSV_PATH is None:
    raise FileNotFoundError(
        f'{_fname} not found.  '
        f'Place it in the data/ subdirectory next to this notebook.')

CONSTRAINTS = ['S', 'A', 'R', 'I', 'H', 'L']
CNAMES = {
    'S': 'Stratification', 'A': 'Acyclicity',  'R': 'Regularity',
    'I': 'Interleaving',   'H': 'Homogeneity', 'L': 'Locality',
}
COLORS6 = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6', '#1abc9c']
CCOLORS = dict(zip(CONSTRAINTS, COLORS6))

raw = pd.read_csv(CSV_PATH)
df  = raw[['Architecture', 'S', 'A', 'R', 'I', 'H', 'L', 'Best_Score']].copy()
assert len(df) == 64, f'Expected 64 rows, got {len(df)}'

# ── Discretisation thresholds — USER ACTION REQUIRED ────────────────────────
# Run Step 0 first to inspect the score distribution, then set these values:
#
#   THRESH_TOP   — Fit_top  (score > threshold)
#     Target: split the 64 architectures approximately 50 / 50 (≈ 32 / 32).
#     A natural break just above the median works well.
#
#   THRESH_ELITE — Fit_elite (score >= threshold)
#     Target: top 10–15 % of architectures (≈ 6 to 10 out of 64).
#     Use the score of the 10th-best architecture, or a natural break
#     in the upper tail of the distribution.
#
THRESH_TOP   = 13.75  # <── adjust: Fit_top   score > threshold  (~50 %,   32 / 64)
THRESH_ELITE = 14.5   # <── adjust: Fit_elite score >= threshold  (~15 %,  10 / 64)

df['Fit_top']   = (df['Best_Score'] >  THRESH_TOP).astype(int)
df['Fit_elite'] = (df['Best_Score'] >= THRESH_ELITE).astype(int)

print(f'CSV loaded    : {CSV_PATH}')
print(f'Loaded        : {len(df)} architectures')
print(f'Score range   : [{df["Best_Score"].min():.4f}, {df["Best_Score"].max():.4f}]  '
      f'(continuous floats)')
print(f'Fit_top   (> {THRESH_TOP})  : {df["Fit_top"].sum():2d} architectures  '
      f'({df["Fit_top"].mean()*100:.1f}%)')
print(f'Fit_elite (>= {THRESH_ELITE}) : {df["Fit_elite"].sum():2d} architectures  '
      f'({df["Fit_elite"].mean()*100:.1f}%)')
generic = df[(df[CONSTRAINTS] == 0).all(axis=1)]
g_score = generic['Best_Score'].values[0]
print(f'Generic (all=0): score = {g_score:.4f}  rank = {generic["Best_Score"].rank(ascending=False).values[0]:.0f}/64  '
      f'({(df["Best_Score"] < g_score).mean()*100:.1f}th percentile)')

---
## Step 0 — Completeness Check & Multi-level Discretisation

> Reminder: adapt `Fit_top` and `Fit_elite` to the context.

`Best_Score` takes **continuous float** values in [12.0, 15.0] with 34 distinct values.
This is a qualitative change from previous runs (which had integer scores with a clear
majority plateau at 8). The improved generator produces a **quasi-continuous, unimodal**
distribution centred around the median.

Thresholds anchored on user-specified natural breaks:

| Column | Rule | Count | Share | Rationale |
|---|---|---|---|---|
| `Fit_top`   | score **> 13.75** | 32 / 64 | 50.0% | Median split (Fit_bin = 50%) |
| `Fit_elite` | score **≥ 14.5**  | 10 / 64 | 15.6% | Exact top-10 cut |

> `Fit_top` is a strict median split — 32 architectures each side.

---
### 0.2 — Score distribution

In [ ]:
desc = df['Best_Score'].describe(percentiles=[.10, .25, .50, .75, .90, .95])
print("Best_Score — descriptive statistics")
print(desc.to_string())
print()
print(f"Distinct values : {df['Best_Score'].nunique()}")
print(f"All integers?   : {all(df['Best_Score'] == df['Best_Score'].astype(int))}")
print()
# Constraint presence in Fit_elite vs all
print(f'Constraint presence rates:')
print(f'  {"":4s}  {"All":>6s}  {"Fit_top":>8s}  {"Fit_elite":>10s}')
for c in CONSTRAINTS:
    a = df[c].mean()
    t = df[df.Fit_top==1][c].mean()
    e = df[df.Fit_elite==1][c].mean()
    print(f'  {c}:   {a:.2f}    {t:.2f}      {e:.2f}')

In [ ]:
# ── Continuous histogram ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 4))
bins = np.linspace(df['Best_Score'].min() - 0.1, df['Best_Score'].max() + 0.1, 30)

# Colour bars by zone
n_all, bin_edges, patches = ax.hist(df['Best_Score'], bins=bins,
                                     edgecolor='white', linewidth=0.8, color='#bdc3c7')
for patch, left in zip(patches, bin_edges[:-1]):
    mid = left + (bin_edges[1] - bin_edges[0]) / 2
    if mid >= THRESH_ELITE:
        patch.set_facecolor('#c0392b')
    elif mid > THRESH_TOP:
        patch.set_facecolor('#e67e22')
    else:
        patch.set_facecolor('#bdc3c7')

ax.axvline(THRESH_TOP,   color='#e67e22', linestyle='--', linewidth=1.6,
           label=f'Fit_top boundary (>{THRESH_TOP}, n={df["Fit_top"].sum()})')
ax.axvline(THRESH_ELITE, color='#c0392b', linestyle='--', linewidth=1.6,
           label=f'Fit_elite boundary (≥{THRESH_ELITE}, n={df["Fit_elite"].sum()})')

# Annotate Generic
g_score = df[(df[CONSTRAINTS] == 0).all(axis=1)]['Best_Score'].values[0]
ax.axvline(g_score, color='#2c3e50', linestyle=':', linewidth=1.4,
           label=f'Generic (all=0) = {g_score:.4f}')

ax.set_xlabel('Best_Score', fontsize=11)
ax.set_ylabel('Count', fontsize=11)
ax.set_title(
    'Step 0 — Best_Score distribution — algebraic_resistance v3 (improved generator)\n'
    '(population=10, generations=90  |  continuous floats in [12, 15])',
    fontsize=11)
ax.legend(fontsize=9, loc='upper left')
plt.tight_layout()
plt.show()

---
### 0.3 — Compute discretised columns

In [ ]:
df['Fit_top']   = (df['Best_Score'] >  THRESH_TOP).astype(int)
df['Fit_elite'] = (df['Best_Score'] >= THRESH_ELITE).astype(int)

print(f"Fit_top   (score > {THRESH_TOP})  : {df['Fit_top'].sum():2d} architectures  "
      f"({df['Fit_top'].mean()*100:.1f}%)")
print(f"Fit_elite (score >= {THRESH_ELITE}) : {df['Fit_elite'].sum():2d} architectures  "
      f"({df['Fit_elite'].mean()*100:.1f}%)")
n_top_only = ((df['Fit_top']==1) & (df['Fit_elite']==0)).sum()
print(f"Fit_top=1 & Fit_elite=0 : {n_top_only} architectures")
g = df[(df[CONSTRAINTS] == 0).all(axis=1)].iloc[0]
print(f"Generic: Best_Score={g['Best_Score']:.4f}  "
      f"Fit_top={g['Fit_top']:.0f}  Fit_elite={g['Fit_elite']:.0f}")

---
### 0.4 — Summary table

In [ ]:
df_display = df.sort_values('Best_Score', ascending=False).reset_index(drop=True)
df_display.index += 1

def row_color(row):
    if row['Fit_elite'] == 1:   bg = '#fadbd8'
    elif row['Fit_top'] == 1:   bg = '#fde8d8'
    else:                       bg = '#fbfcfc'
    return [f'background-color: {bg}'] * len(row)

display(
    df_display.style
    .apply(row_color, axis=1)
    .format({'Best_Score': '{:.4f}', **{c: '{:.0f}' for c in CONSTRAINTS},
             'Fit_top': '{:.0f}', 'Fit_elite': '{:.0f}'})
    .set_caption(
        f'Step 0 — 64 architectures sorted by Best_Score. '
        f'Red = elite (≥{THRESH_ELITE}, top 10) | Orange = Fit_top (>{THRESH_TOP}, top 50%).')
    .set_table_styles([
        {'selector':'caption','props':[('font-size','12px'),('font-weight','bold'),
                                        ('padding','8px 0'),('text-align','left')]},
        {'selector':'th','props':[('background-color','#2c3e50'),('color','white'),
                                   ('font-size','12px'),('padding','6px 12px')]},
        {'selector':'td','props':[('font-size','11px'),('padding','4px 12px'),
                                   ('text-align','center')]}])
)

---
## Step 1 — Hypercube Edge Analysis

$$\Delta_{C_i}(\text{context}) = f(\ldots, C_i=1, \ldots) - f(\ldots, C_i=0, \ldots)$$

32 edges per constraint, one Generic-vertex edge flagged separately.
Scores are continuous floats — delta values shown as `.2f`.

---
### 1.2 — Build edge table

In [ ]:
records = []
for ci in CONSTRAINTS:
    others = [c for c in CONSTRAINTS if c != ci]
    for _, grp in df.groupby(others):
        if len(grp) != 2: continue
        row0 = grp[grp[ci] == 0].iloc[0]
        row1 = grp[grp[ci] == 1].iloc[0]
        delta    = float(row1['Best_Score'] - row0['Best_Score'])
        ctx_int  = sum(int(row0[c]) << i for i, c in enumerate(others))
        from_gen = sum(int(row0[c]) for c in CONSTRAINTS if c != ci) == 0
        records.append({
            'constraint': ci, 'delta': delta,
            'ctx_int': ctx_int, 'from_generic': from_gen,
            'score_0': float(row0['Best_Score']),
            'score_1': float(row1['Best_Score']),
            'arch_0': row0['Architecture'], 'arch_1': row1['Architecture'],
            **{c: int(row0[c]) for c in others},
        })

edges = pd.DataFrame(records)
print(f'Total edges : {len(edges)}  (expected 192 = 6 × 32)')
print(f'Generic-vertex edges : {edges["from_generic"].sum()}')
print()
print('Generic-vertex edge deltas:')
for _, r in edges[edges['from_generic']].iterrows():
    print(f'  Activating {r["constraint"]}: {r["score_0"]:.4f} → '
          f'{r["score_1"]:.4f}   Δ = {r["delta"]:+.4f}')
print()
print(f'{"C":>2}  {"mean Δ":>9}  {"std Δ":>9}  {"CV":>8}  {"+":>4}  {"0":>5}  {"-":>4}')
print('-' * 58)
for ci in CONSTRAINTS:
    sub = edges[(edges['constraint'] == ci) & (~edges['from_generic'])]['delta']
    mu  = sub.mean(); sig = sub.std()
    cv  = sig / abs(mu) if abs(mu) > 1e-9 else float('inf')
    print(f'{ci:>2}  {mu:>9.4f}  {sig:>9.4f}  '
          f'{"inf" if np.isinf(cv) else f"{cv:.2f}":>8}  '
          f'{(sub>0).sum():>4}  {(sub==0).sum():>5}  {(sub<0).sum():>4}')

---
### 1.3 — Histograms of ΔCᵢ

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
axes = axes.flatten()
d_min = edges['delta'].min(); d_max = edges['delta'].max()
bins  = np.linspace(d_min - 0.05, d_max + 0.05, 28)

for ax, ci in zip(axes, CONSTRAINTS):
    sub   = edges[edges['constraint'] == ci]
    nogen = sub[~sub['from_generic']]['delta'].values
    gen   = sub[sub['from_generic']]['delta'].values
    mu  = nogen.mean(); sig = nogen.std()
    cv  = sig / abs(mu) if abs(mu) > 1e-9 else float('inf')

    ax.hist(nogen, bins=bins, color=CCOLORS[ci], alpha=0.75,
            edgecolor='white', linewidth=0.6, label='Regular edges')
    for dg in gen:
        ax.axvline(dg, color='black', linewidth=1.8, linestyle='--', alpha=0.9,
                   label=f'Generic ({dg:+.3f})')
    ax.axvline(0,  color='#555', linewidth=0.9, linestyle=':')
    ax.axvline(mu, color=CCOLORS[ci], linewidth=1.6, linestyle='-', alpha=0.9)

    pos  = (nogen > 1e-6).sum(); zero = (abs(nogen) <= 1e-6).sum(); neg = (nogen < -1e-6).sum()
    cv_s = f'{cv:.2f}' if not np.isinf(cv) else 'inf'
    ax.set_title(f'{ci} — {CNAMES[ci]}', fontsize=11, fontweight='bold')
    ax.set_xlabel('Δ fitness', fontsize=9); ax.set_ylabel('Count', fontsize=9)
    ax.legend(fontsize=7.5, loc='upper left')
    ax.text(0.98, 0.97, f'+{pos} / 0:{zero} / −{neg}\nμ={mu:+.3f}  CV={cv_s}',
            transform=ax.transAxes, fontsize=8, va='top', ha='right',
            bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.8))

plt.suptitle('Step 1 — Marginal effect ΔCᵢ distributions  (continuous float scores)\n'
             '(31 regular edges + 1 Generic edge per constraint)',
             fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

---
### 1.4 — Sign heatmap  (32 × 6)

In [ ]:
sign_data  = np.zeros((32, 6), dtype=float)
delta_data = np.zeros((32, 6), dtype=float)
generic_rows = []

for col_idx, ci in enumerate(CONSTRAINTS):
    sub = (edges[edges['constraint'] == ci]
           .sort_values('ctx_int').reset_index(drop=True))
    sign_data[:, col_idx]  = np.sign(sub['delta'].values)
    delta_data[:, col_idx] = sub['delta'].values
    gr = sub[sub['from_generic']].index.tolist()
    generic_rows.append(gr[0] if gr else None)

cmap = mcolors.LinearSegmentedColormap.from_list(
    'sign', ['#e74c3c', '#f5f5f5', '#27ae60'], N=256)

fig, ax = plt.subplots(figsize=(7, 11))
ax.imshow(sign_data, cmap=cmap, vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(6))
ax.set_xticklabels([f'{ci}\n{CNAMES[ci]}' for ci in CONSTRAINTS], fontsize=9)
ax.set_yticks(range(32))
ax.set_yticklabels([f'{i:02d}' for i in range(32)], fontsize=7)
ax.set_ylabel('Context index (sorted by ctx_int)', fontsize=9)
ax.set_title('Step 1 — Sign of ΔCᵢ across 32 contexts\n'
             '(green = +, white = 0, red = −)', fontsize=11, pad=10)

for col_idx in range(6):
    for row_idx in range(32):
        delta = delta_data[row_idx, col_idx]
        ax.text(col_idx, row_idx, f'{delta:+.2f}',
                ha='center', va='center', fontsize=6.5, color='black')
    gr = generic_rows[col_idx]
    if gr is not None:
        ax.add_patch(plt.Rectangle((col_idx-0.5, gr-0.5), 1, 1,
                                   fill=False, edgecolor='gold', linewidth=2.5))

ax.legend(handles=[
    mpatches.Patch(color='#27ae60', label='ΔCᵢ > 0'),
    mpatches.Patch(color='#f5f5f5', label='ΔCᵢ ≈ 0', edgecolor='#aaa'),
    mpatches.Patch(color='#e74c3c', label='ΔCᵢ < 0'),
    mpatches.Patch(color='white', label='Gold = Generic edge', edgecolor='gold', linewidth=2),
], fontsize=8, loc='lower right', bbox_to_anchor=(1.0, -0.08), ncol=2)
plt.tight_layout()
plt.show()

---
### 1.5 — Stability : Coefficient of Variation

In [ ]:
cv_rows = []
for ci in CONSTRAINTS:
    sub = edges[(edges['constraint'] == ci) & (~edges['from_generic'])]['delta']
    mu = sub.mean(); sig = sub.std()
    cv = sig / abs(mu) if abs(mu) > 1e-9 else np.inf
    cv_rows.append({'Constraint': ci, 'Name': CNAMES[ci],
                    'mu': mu, 'sigma': sig, 'CV': cv,
                    '+': (sub > 1e-6).sum(),
                    '0': (abs(sub) <= 1e-6).sum(),
                    '-': (sub < -1e-6).sum()})
cv_df = pd.DataFrame(cv_rows).sort_values('CV')

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

ax = axes[0]
sor   = cv_df['Constraint'].tolist()
means = cv_df['mu'].values; stds = cv_df['sigma'].values
cols  = [CCOLORS[ci] for ci in sor]
ax.barh(sor, means, xerr=stds, color=cols, alpha=0.82, edgecolor='white',
        error_kw=dict(ecolor='#333', capsize=4, linewidth=1.2))
ax.axvline(0, color='#555', linewidth=0.9, linestyle=':')
for i, (m, s) in enumerate(zip(means, stds)):
    ax.text(m + np.sign(m)*(s+0.005), i, f'{m:+.3f}', va='center',
            ha='left' if m>=0 else 'right', fontsize=8)
ax.set_xlabel('Mean marginal effect  μ(ΔCᵢ)', fontsize=10)
ax.set_title('Mean Δ ± 1σ  (31 regular edges)', fontsize=10)
ax.invert_yaxis()

ax2 = axes[1]
cvs = [min(v, 110) if not np.isinf(v) else 110 for v in cv_df['CV'].values]
ax2.barh(sor, cvs, color=cols, alpha=0.82, edgecolor='white')
ax2.axvspan(0,   1, alpha=0.08, color='#27ae60')
ax2.axvspan(1,   3, alpha=0.08, color='#f39c12')
ax2.axvspan(3, 115, alpha=0.08, color='#e74c3c')
for i, (v, ci) in enumerate(zip(cv_df['CV'].values, sor)):
    ax2.text(min(v,110)+1, i, f'{v:.2f}' if not np.isinf(v) else 'inf',
             va='center', fontsize=8)
ax2.set_xlabel('CV = σ / |μ|', fontsize=10)
ax2.set_title('Stability of marginal effect  (lower = more stable)', fontsize=10)
ax2.set_xlim(0, 115); ax2.invert_yaxis()
ax2.legend(handles=[
    mpatches.Patch(color='#27ae60', alpha=0.4, label='CV < 1  — stable'),
    mpatches.Patch(color='#f39c12', alpha=0.4, label='1 ≤ CV < 3  — moderate'),
    mpatches.Patch(color='#e74c3c', alpha=0.4, label='CV ≥ 3  — epistatic'),
], fontsize=8, loc='lower right')
plt.suptitle('Step 1 — Marginal effect summary & stability  (Generic vertex excluded)',
             fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

dtab = cv_df[['Constraint','Name','+','0','-','CV']].copy()
dtab.columns = ['C','Name','+','0','-','CV']
dtab['CV'] = dtab['CV'].apply(lambda v: 'inf' if np.isinf(v) else f'{v:.2f}')
def style_cv(val):
    try: v = float(val)
    except: return 'background-color:#fdecea'
    if v < 1: return 'background-color:#d5f5e3'
    if v < 3: return 'background-color:#fef9e7'
    return 'background-color:#fdecea'
display(dtab.style.map(style_cv, subset=['CV'])
    .set_caption('Step 1 — Stability table (sorted by CV). Generic edge excluded.')
    .set_table_styles([
        {'selector':'th','props':[('background-color','#2c3e50'),('color','white'),
                                   ('padding','6px 12px')]},
        {'selector':'td','props':[('text-align','center'),('padding','4px 12px')]}]))

---
## Step 2 — Epistasis Matrix

$$\varepsilon(C_i, C_j, \text{ctx}) = f(1,1) - f(1,0) - f(0,1) + f(0,0)$$

Raw continuous `Best_Score`. 16 unit squares per pair.

---
### 2.2 — Compute epistasis for all 15 pairs

In [ ]:
N = len(CONSTRAINTS)
mat_mean = np.full((N, N), np.nan)
mat_std  = np.full((N, N), np.nan)
mat_snr  = np.full((N, N), np.nan)
all_eps  = {}

pairs = [(CONSTRAINTS[i], CONSTRAINTS[j])
         for i in range(N) for j in range(i+1, N)]

for ci, cj in pairs:
    ii, jj = CONSTRAINTS.index(ci), CONSTRAINTS.index(cj)
    others  = [c for c in CONSTRAINTS if c not in (ci, cj)]
    eps = []
    for _, grp in df.groupby(others):
        if len(grp) != 4: continue
        f00 = grp[(grp[ci]==0)&(grp[cj]==0)]['Best_Score'].values
        f10 = grp[(grp[ci]==1)&(grp[cj]==0)]['Best_Score'].values
        f01 = grp[(grp[ci]==0)&(grp[cj]==1)]['Best_Score'].values
        f11 = grp[(grp[ci]==1)&(grp[cj]==1)]['Best_Score'].values
        if all(len(x)==1 for x in [f00,f10,f01,f11]):
            eps.append(float(f11[0]-f10[0]-f01[0]+f00[0]))
    mu  = np.mean(eps); sig = np.std(eps)
    snr = abs(mu)/sig if sig > 1e-9 else 0.0
    for r, c in [(ii,jj),(jj,ii)]:
        mat_mean[r,c]=mu; mat_std[r,c]=sig; mat_snr[r,c]=snr
    all_eps[(ci,cj)] = eps

rows = []
for ci, cj in pairs:
    ii, jj = CONSTRAINTS.index(ci), CONSTRAINTS.index(cj)
    rows.append({'Pair': f'{ci}+{cj}',
                 'Mean ε': round(mat_mean[ii,jj], 4),
                 'Std ε':  round(mat_std[ii,jj],  4),
                 'SNR':    round(mat_snr[ii,jj],   3),
                 'Synergy': sum(1 for e in all_eps[(ci,cj)] if e >  1e-6),
                 'Neutral': sum(1 for e in all_eps[(ci,cj)] if abs(e) <= 1e-6),
                 'Antag.':  sum(1 for e in all_eps[(ci,cj)] if e < -1e-6)})
summary = pd.DataFrame(rows).sort_values('SNR', ascending=False)
print(f'Pairs: {len(pairs)}/15  |  16 contexts each')
print()
print(summary.to_string(index=False))

---
### 2.3 — Epistasis matrices: Mean ε and SNR

In [ ]:
labels_s = CONSTRAINTS
vmax_m   = max(np.nanmax(np.abs(mat_mean)), 0.01)
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

ax = axes[0]
masked_mean = np.where(np.eye(N,dtype=bool), np.nan, mat_mean)
im = ax.imshow(masked_mean, cmap='RdYlGn',
               norm=TwoSlopeNorm(vmin=-vmax_m, vcenter=0, vmax=vmax_m))
for i in range(N):
    for j in range(N):
        if i==j:
            ax.add_patch(plt.Rectangle((j-.5,i-.5),1,1,color='#ccc',zorder=2))
            ax.text(j,i,'---',ha='center',va='center',fontsize=9,color='#888',zorder=3)
        else:
            val = mat_mean[i,j]
            ax.text(j,i,f'{val:+.3f}',ha='center',va='center',fontsize=12,
                    fontweight='bold',
                    color='white' if abs(val)>vmax_m*0.55 else 'black')
ax.set_xticks(range(N)); ax.set_xticklabels(labels_s, fontsize=11)
ax.set_yticks(range(N))
ax.set_yticklabels([f'{c} {CNAMES[c]}' for c in CONSTRAINTS], fontsize=9)
#ax.set_title('Mean epistasis  μ(ε)\n(green=synergy, red=antagonism)', fontsize=10, pad=10)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='Mean ε')

ax2 = axes[1]
masked_snr = np.where(np.eye(N,dtype=bool), np.nan, mat_snr)
im2 = ax2.imshow(masked_snr, cmap='YlOrRd', vmin=0,
                 vmax=max(1.0, float(np.nanmax(mat_snr))))
for i in range(N):
    for j in range(N):
        if i==j:
            ax2.add_patch(plt.Rectangle((j-.5,i-.5),1,1,color='#ccc',zorder=2))
            ax2.text(j,i,'---',ha='center',va='center',fontsize=9,color='#888',zorder=3)
        else:
            val = mat_snr[i,j]
            ax2.text(j,i,f'{val:.2f}',ha='center',va='center',fontsize=8,
                     fontweight='bold',color='white' if val>0.7 else 'black')
ax2.set_xticks(range(N)); ax2.set_xticklabels(labels_s, fontsize=11)
ax2.set_yticks(range(N))
ax2.set_yticklabels([f'{c} {CNAMES[c]}' for c in CONSTRAINTS], fontsize=9)
ax2.set_title('SNR = |μ|/σ\n(higher = more reliable)', fontsize=10, pad=10)
plt.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04, label='SNR')
plt.suptitle('Step 2 — Epistasis Matrix  (symmetric, 16 contexts per pair)',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

---
### 2.4 — Std ε and strength vs variability scatter

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

ax = axes[0]
masked_std = np.where(np.eye(N,dtype=bool), np.nan, mat_std)
im = ax.imshow(masked_std, cmap='Blues', vmin=0)
for i in range(N):
    for j in range(N):
        if i==j:
            ax.add_patch(plt.Rectangle((j-.5,i-.5),1,1,color='#ccc',zorder=2))
            ax.text(j,i,'---',ha='center',va='center',fontsize=9,color='#888',zorder=3)
        else:
            val = mat_std[i,j]
            ax.text(j,i,f'{val:.3f}',ha='center',va='center',fontsize=8,
                    fontweight='bold',
                    color='white' if val>float(np.nanmax(masked_std))*0.65 else 'black')
ax.set_xticks(range(N)); ax.set_xticklabels(labels_s, fontsize=11)
ax.set_yticks(range(N))
ax.set_yticklabels([f'{c} {CNAMES[c]}' for c in CONSTRAINTS], fontsize=9)
ax.set_title('Std of epistasis  σ(ε)', fontsize=10, pad=10)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='Std ε')

ax2 = axes[1]
for ci, cj in pairs:
    ii, jj = CONSTRAINTS.index(ci), CONSTRAINTS.index(cj)
    mu_v=mat_mean[ii,jj]; sig_v=mat_std[ii,jj]; snr_v=mat_snr[ii,jj]
    ax2.scatter(sig_v, abs(mu_v), s=snr_v*400+40,
                color=COLORS6[ii], edgecolors=COLORS6[jj],
                linewidths=2.5, alpha=0.85, zorder=3)
    ax2.annotate(f'{ci}+{cj}', (sig_v, abs(mu_v)),
                 textcoords='offset points', xytext=(5,3), fontsize=7.5, color='#333')
sig_range = np.linspace(0.005, float(np.nanmax(mat_std))*1.1+0.01, 200)
for snr_iso, ls in [(0.25,':'),(0.5,'--'),(0.9,'-')]:
    ax2.plot(sig_range, snr_iso*sig_range, color='#aaa', linestyle=ls,
             linewidth=1, label=f'SNR={snr_iso}')
ax2.set_xlabel('σ(ε)', fontsize=10); ax2.set_ylabel('|μ(ε)|', fontsize=10)
ax2.set_title('Interaction strength vs variability\n(fill=Ci, border=Cj, size∝SNR)',
              fontsize=10)
ax2.legend(title='Iso-SNR', fontsize=8, title_fontsize=8, loc='upper left')
ax2.grid(True, alpha=0.3)
fig.legend(handles=[mpatches.Patch(color=COLORS6[i],
           label=f'{CONSTRAINTS[i]} {CNAMES[CONSTRAINTS[i]]}') for i in range(N)],
           fontsize=7.5, loc='lower center', ncol=3,
           title='Constraint colour', title_fontsize=8, bbox_to_anchor=(0.5,-0.05))
plt.suptitle('Step 2 — Context variability of epistatic interactions',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

---
### 2.5 — ε distribution (6 highest-SNR pairs)

In [ ]:
top_pairs_t = [(p.split('+')[0], p.split('+')[1])
               for p in summary.head(6)['Pair'].tolist()]

fig, axes = plt.subplots(2, 3, figsize=(13, 6.5))
axes = axes.flatten()

for ax, (ci,cj) in zip(axes, top_pairs_t):
    eps_vals = all_eps[(ci,cj)]
    mu_v  = mat_mean[CONSTRAINTS.index(ci), CONSTRAINTS.index(cj)]
    snr_v = mat_snr[CONSTRAINTS.index(ci),  CONSTRAINTS.index(cj)]
    bins  = np.linspace(min(eps_vals)-0.05, max(eps_vals)+0.05, 16)
    counts_h, edges_ = np.histogram(eps_vals, bins=bins)
    for c, lo, hi in zip(counts_h, edges_[:-1], edges_[1:]):
        mid = (lo+hi)/2
        col = '#27ae60' if mid>1e-6 else '#e74c3c' if mid<-1e-6 else '#95a5a6'
        ax.bar((lo+hi)/2, c, width=(hi-lo)*0.85, color=col, alpha=0.8, edgecolor='white')
    ax.axvline(0,    color='#555', linewidth=0.9, linestyle=':')
    ax.axvline(mu_v, color='#2c3e50', linewidth=1.8, linestyle='--',
               label=f'μ={mu_v:+.3f}')
    n_syn  = sum(1 for e in eps_vals if e >  1e-6)
    n_zero = sum(1 for e in eps_vals if abs(e) <= 1e-6)
    n_ant  = sum(1 for e in eps_vals if e < -1e-6)
    ax.set_title(f'{ci}+{cj}', fontsize=10, fontweight='bold')
    ax.set_xlabel('ε', fontsize=8); ax.set_ylabel('Count', fontsize=8)
    ax.legend(fontsize=8)
    ax.text(0.98, 0.97, f'SNR={snr_v:.2f}\n+{n_syn}/0:{n_zero}/−{n_ant}',
            transform=ax.transAxes, fontsize=8, va='top', ha='right',
            bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.85))
plt.suptitle('Step 2 — ε distribution across 16 contexts  (6 highest-SNR pairs)',
             fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

---
### 2.6 — Summary table

In [ ]:
def style_mean(val):
    try: v = float(val)
    except: return ''
    if v < -0.5: return 'background-color:#fdecea;color:#922b21'
    if v >  0.5: return 'background-color:#d5f5e3;color:#1e8449'
    return ''
def style_snr(val):
    try: v = float(val)
    except: return ''
    if v >= 0.7: return 'background-color:#d5f5e3;font-weight:bold'
    if v >= 0.4: return 'background-color:#fef9e7'
    return ''
display(
    summary.style
    .map(style_mean, subset=['Mean ε'])
    .map(style_snr,  subset=['SNR'])
    .set_caption('Step 2 — Epistasis summary (sorted by SNR). Continuous float scores.')
    .set_table_styles([
        {'selector':'caption','props':[('font-size','12px'),('font-weight','bold'),
                                        ('padding','6px 0'),('text-align','left')]},
        {'selector':'th','props':[('background-color','#2c3e50'),('color','white'),
                                   ('font-size','12px'),('padding','6px 14px')]},
        {'selector':'td','props':[('font-size','12px'),('padding','4px 14px'),
                                   ('text-align','center')]}])
)

---
## Step 3 — FCA Lattice for Minimal Rules

FCA operates on the binary discretisation of the continuous scores.

Two contexts:
1. `Fit_top = 1`   (score > 13.75, n=32) — top 50% by median split  
2. `Fit_elite = 1` (score ≥ 14.5,  n=10) — exact top-10  

Implementation: Stem Base (Duquenne–Guigues basis), confidence = 1.0 by construction.

In [ ]:
from itertools import combinations
import time, pandas as _pd, os as _os

ATTRS = ['S', 'A', 'R', 'I', 'H', 'L']

def obj_cl(X, ctx):
    X = frozenset(X)
    ext = [a for _, a in ctx if X <= a]
    if not ext: return frozenset(ATTRS)
    sh = set(ext[0])
    for a in ext[1:]: sh &= a
    return frozenset(sh)

def all_closed_sets(ctx):
    closed = set()
    for r in range(len(ATTRS)+1):
        for combo in combinations(ATTRS, r):
            X = frozenset(combo)
            if obj_cl(X, ctx) == X:
                closed.add(X)
    return closed

def compute_stem_base(ctx):
    all_sub = [frozenset(c) for r in range(len(ATTRS)+1)
               for c in combinations(ATTRS, r)]
    closed = all_closed_sets(ctx)
    pseudo_closed = []
    for P in all_sub:
        if P in closed: continue
        if all(obj_cl(Q, ctx) <= P for Q in pseudo_closed if Q < P):
            conclusion = obj_cl(P, ctx) - P
            if conclusion: pseudo_closed.append(P)
    return [(P, obj_cl(P, ctx)-P) for P in pseudo_closed]

def fmt_rule(P, C):
    return ('+'.join(sorted(P)) if P else '∅') + '  →  ' + '+'.join(sorted(C))

def rule_stats(impl, ctx):
    n = len(ctx)
    if n == 0: return []
    return [{'Rule': fmt_rule(P,C), 'Premise': sorted(P), 'Conclusion': sorted(C),
             'Support':  round(sum(1 for _,a in ctx if (P|C)<=a)/n, 3),
             'Coverage': sum(1 for _,a in ctx if P<=a),
             'Confidence': 1.0}
            for P,C in impl]

# ── Build contexts — reuse CSV_PATH and thresholds from Shared Setup ─────────
_df = _pd.read_csv(CSV_PATH)
print(f'CSV loaded: {len(_df)} rows  |  {CSV_PATH}')

ctx_top   = [(row['Architecture'], frozenset(c for c in ATTRS if row[c]==1))
             for _, row in _df[_df['Best_Score'] >  THRESH_TOP].iterrows()]
ctx_elite = [(row['Architecture'], frozenset(c for c in ATTRS if row[c]==1))
             for _, row in _df[_df['Best_Score'] >= THRESH_ELITE].iterrows()]
ctx_full  = [(row['Architecture'], frozenset(c for c in ATTRS if row[c]==1))
             for _, row in _df.iterrows()]

t0 = time.time()
closed_top   = all_closed_sets(ctx_top)   if ctx_top   else set()
closed_elite = all_closed_sets(ctx_elite) if ctx_elite else set()
impl_top     = compute_stem_base(ctx_top)   if ctx_top   else []
impl_elite   = compute_stem_base(ctx_elite) if ctx_elite else []
elapsed = time.time() - t0

rows_top   = rule_stats(impl_top,   ctx_top)
rows_elite = rule_stats(impl_elite, ctx_elite)

print(f'Computed in {elapsed:.3f}s')
print(f'Fit_top   (n={len(ctx_top)})  : {len(closed_top)} closed sets, '
      f'{len(impl_top)} implications')
print(f'Fit_elite (n={len(ctx_elite)}) : {len(closed_elite)} closed sets, '
      f'{len(impl_elite)} implications')
print()
print('STEM BASE — Fit_top')
for r in sorted(rows_top, key=lambda x: -x['Support']):
    print(f"  {r['Rule']:<50}  supp={r['Support']:.3f}  cov={r['Coverage']}")
print()
print('STEM BASE — Fit_elite')
for r in sorted(rows_elite, key=lambda x: -x['Support']):
    print(f"  {r['Rule']:<50}  supp={r['Support']:.3f}  cov={r['Coverage']}")

In [ ]:
# ── Artefact 1 : Formal context heatmaps ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 9))

for ax, ctx, title in [
    (axes[0], ctx_top,   f'Fit_top = 1  (n={len(ctx_top)}, score > 13.75)'),
    (axes[1], ctx_elite, f'Fit_elite = 1  (n={len(ctx_elite)}, score ≥ 14.5)'),
]:
    if not ctx:
        ax.text(0.5, 0.5, 'Empty context', ha='center', va='center',
                transform=ax.transAxes, fontsize=12)
        ax.set_title(title, fontsize=10); continue
    n_obj = len(ctx)
    matrix = np.zeros((n_obj, len(ATTRS)), dtype=int)
    scores = {row['Architecture']: row['Best_Score'] for _, row in _df.iterrows()}
    ctx_s  = sorted(ctx, key=lambda x: -scores.get(x[0], 0))
    labels_obj = []
    for i, (obj, attrs) in enumerate(ctx_s):
        labels_obj.append(f"{obj} ({scores.get(obj,0):.4f})")
        for j, a in enumerate(ATTRS):
            matrix[i,j] = 1 if a in attrs else 0

    ax.imshow(matrix, cmap='Blues', vmin=0, vmax=1.3, aspect='auto')
    ax.set_xticks(range(len(ATTRS))); ax.set_xticklabels(ATTRS, fontsize=12, fontweight='bold')
    ax.set_yticks(range(n_obj)); ax.set_yticklabels(labels_obj, fontsize=7)
    ax.set_title(title, fontsize=10, pad=10)
    for i in range(n_obj):
        for j in range(len(ATTRS)):
            val = matrix[i,j]
            ax.text(j, i, '●' if val else '·', ha='center', va='center',
                    fontsize=11 if val else 8, color='white' if val else '#bbb')

plt.suptitle('Step 3 — Formal contexts: constraint presence (●) by architecture',
             fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Artefact 2 : Implication rules ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

for ax, rows, title in [
    (axes[0], rows_top,   f'Fit_top rules  (n={len(ctx_top)})'),
    (axes[1], rows_elite, f'Fit_elite rules  (n={len(ctx_elite)})'),
]:
    if not rows:
        ax.text(0.5, 0.5, 'No non-trivial rules', ha='center', va='center',
                transform=ax.transAxes, fontsize=12)
        ax.set_title(title, fontsize=10); continue
    rows_s = sorted(rows, key=lambda x: (-x['Support'], len(x['Premise'])))
    labels = [r['Rule'] for r in rows_s]
    supps  = [r['Support'] for r in rows_s]
    covs   = [r['Coverage'] for r in rows_s]
    bar_cols = ['#c0392b' if s>=0.5 else '#e67e22' if s>=0.25 else '#f1c40f'
                for s in supps]
    y = range(len(labels))
    ax.barh(list(y), supps, color=bar_cols, alpha=0.85, edgecolor='white')
    for i, (supp, cov) in enumerate(zip(supps, covs)):
        ax.text(supp+0.01, i, f'{supp:.3f}  (cov={cov})', va='center', fontsize=8)
    ax.set_yticks(list(y)); ax.set_yticklabels(labels, fontsize=8)
    ax.set_xlabel('Support', fontsize=9)
    ax.set_xlim(0, 1.35); ax.set_title(title, fontsize=10, pad=8)
    ax.invert_yaxis()
    for thresh, col, ls in [(0.25,'#f1c40f',':'),(0.5,'#e67e22','--'),(1.0,'#c0392b','-')]:
        ax.axvline(thresh, color=col, linewidth=1.2, linestyle=ls, alpha=0.8)
    ax.legend(handles=[
        mpatches.Patch(color='#c0392b', alpha=0.85, label='supp ≥ 0.50'),
        mpatches.Patch(color='#e67e22', alpha=0.85, label='0.25 ≤ supp < 0.50'),
        mpatches.Patch(color='#f1c40f', alpha=0.85, label='supp < 0.25'),
    ], fontsize=7.5, loc='lower right')

plt.suptitle('Step 3 — Stem base (Duquenne-Guigues)  |  confidence = 1.0 by construction',
             fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Artefact 3 : Hasse diagram — Fit_top context ─────────────────────────────
from collections import defaultdict

def compute_extents(ctx, concepts):
    return {i: frozenset(o for o,a in ctx if i<=a) for i in concepts}

def hasse_edges(concepts):
    cl = sorted(concepts, key=len)
    return [(A,B) for i,A in enumerate(cl) for B in cl[i+1:]
            if A<B and not any(A<C<B for C in cl)], cl

concepts_top = sorted(closed_top, key=len) if closed_top else []
if not concepts_top:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.text(0.5, 0.5, 'No concepts (empty context)', ha='center', va='center',
            transform=ax.transAxes, fontsize=12)
    plt.tight_layout(); plt.show()
else:
    extents_top  = compute_extents(ctx_top, concepts_top)
    hedges, csorted = hasse_edges(concepts_top)
    levels = defaultdict(list)
    for c in csorted: levels[len(c)].append(c)
    pos = {node: ((k-(len(nodes)-1)/2)*2.5, lvl*1.8)
           for lvl, nodes in levels.items()
           for k, node in enumerate(nodes)}

    fig, ax = plt.subplots(figsize=(13, 9))
    for A,B in hedges:
        x0,y0=pos[A]; x1,y1=pos[B]
        ax.plot([x0,x1],[y0,y1], color='#bdc3c7', linewidth=1.2, zorder=1)
    for intent in csorted:
        x,y = pos[intent]; n_ext = len(extents_top[intent])
        radius = 0.35 + n_ext*0.03
        color = ('#2c3e50' if len(intent)==0 else
                 '#e74c3c' if len(intent)==len(ATTRS) else
                 plt.cm.Blues(0.3+0.6*len(intent)/len(ATTRS)))
        ax.add_patch(plt.Circle((x,y), radius, color=color, zorder=2, alpha=0.9))
        lbl = '+'.join(sorted(intent)) if intent else '∅'
        ax.text(x, y+0.05, lbl, ha='center', va='center',
                fontsize=7, color='white', fontweight='bold', zorder=3)
        ax.text(x, y-radius-0.22, f'n={n_ext}', ha='center', va='top',
                fontsize=6.5, color='#555', zorder=3)
    if pos:
        ax.set_xlim(min(p[0] for p in pos.values())-2, max(p[0] for p in pos.values())+2)
        ax.set_ylim(-1, max(p[1] for p in pos.values())+1.5)
    ax.axis('off')
    for lvl in levels:
        ax.text(ax.get_xlim()[0]+0.2, lvl*1.8,
                f'{lvl} attr{"s" if lvl!=1 else ""}',
                va='center', fontsize=8, color='#888', style='italic')
    ax.set_title(f'Step 3 — Concept lattice — Fit_top (n={len(ctx_top)})\n'
                 'Node = intent, n = extent size.', fontsize=11, pad=12)
    plt.tight_layout(); plt.show()

---
## Step 4 — Post-lattice Validation

1. Rule confidence on the full 64-architecture dataset  
2. Single-constraint lift on Fit_top  
3. Best combination ranking — mean score and Fit_elite rate  
4. Coherence matrix — CV (Step 1) × SNR (Step 2) × Lift (Step 4)  

In [ ]:
from itertools import combinations as _comb
import warnings; warnings.filterwarnings('ignore')

# ── 4.1  Rule confidence on full dataset ──────────────────────────────────────
print("=" * 65)
print("4.1 — Rule confidence on the FULL dataset (n=64)")
print("=" * 65)

val_rows = []
for label, impl in [('Fit_top', impl_top), ('Fit_elite', impl_elite)]:
    for P, C in impl:
        cov_full  = sum(1 for _,a in ctx_full if P<=a)
        hits_full = sum(1 for _,a in ctx_full if (P|C)<=a)
        conf_full = hits_full/cov_full if cov_full>0 else 0.0
        val_rows.append({
            'Context'    : label,
            'Rule'       : fmt_rule(P,C),
            'Conf (full)': round(conf_full, 2),
            'Cov (full)' : cov_full,
            'Hits'       : hits_full,
        })

if val_rows:
    val_df = pd.DataFrame(val_rows)
    def style_conf(val):
        try: v = float(val)
        except: return ''
        if v >= 0.75: return 'background-color:#d5f5e3;font-weight:bold'
        if v >= 0.45: return 'background-color:#fef9e7'
        return 'background-color:#fdecea'
    display(
        val_df.style.map(style_conf, subset=['Conf (full)'])
        .set_caption('Step 4.1 — Rule confidence on full dataset.')
        .set_table_styles([
            {'selector':'th','props':[('background-color','#2c3e50'),('color','white'),
                                       ('font-size','12px'),('padding','6px 12px')]},
            {'selector':'td','props':[('font-size','11px'),('padding','4px 12px'),
                                       ('text-align','center')]},
            {'selector':'caption','props':[('font-size','12px'),('font-weight','bold'),
                                            ('padding','6px 0'),('text-align','left')]}])
    )
else:
    print('No implications found.')

In [ ]:
# ── 4.2  Single-constraint lift on Fit_top ────────────────────────────────────
baseline  = df['Fit_top'].mean()
lift_rows = []
for c in ATTRS:
    r1   = df[df[c]==1]['Fit_top'].mean()
    r0   = df[df[c]==0]['Fit_top'].mean()
    lift = r1/r0 if r0>0 else np.inf
    lift_rows.append({'Constraint': c, 'Name': CNAMES[c],
                      'Rate active': round(r1,3), 'Rate inactive': round(r0,3),
                      'Lift': round(lift,3)})
lift_df = pd.DataFrame(lift_rows).sort_values('Lift', ascending=False)

fig, ax = plt.subplots(figsize=(9, 3.5))
lifts = lift_df['Lift'].values
cols  = [CCOLORS[c] for c in lift_df['Constraint']]
ax.barh(lift_df['Constraint'].tolist(), lifts, color=cols, alpha=0.85, edgecolor='white')
ax.axvline(1.0, color='#555', linewidth=1.2, linestyle='--', label='Lift=1.0')
for i, l in enumerate(lifts):
    ax.text((l if not np.isinf(l) else 2.5)+0.02, i,
            f'{l:.2f}x' if not np.isinf(l) else 'inf', va='center', fontsize=9)
finite_l = lifts[np.isfinite(lifts)]
ax.set_xlim(0, max(finite_l.max()+0.4, 1.8) if len(finite_l) else 3)
ax.set_xlabel('Lift  (Fit_top rate: active / inactive)', fontsize=10)
ax.set_title(f'Step 4.2 — Single-constraint lift on Fit_top  '
             f'(baseline: {baseline:.2f} = {df["Fit_top"].sum()}/64)', fontsize=10)
ax.legend(fontsize=9); ax.invert_yaxis()
plt.tight_layout(); plt.show()
print(lift_df.to_string(index=False))

In [ ]:
# ── 4.3  Best constraint combinations ─────────────────────────────────────────
combo_rows = []
g_sub = df[(df[CONSTRAINTS]==0).all(axis=1)]
if len(g_sub):
    combo_rows.append({'Combination':'Generic (∅)', 'n':len(g_sub),
                       'Mean score': round(g_sub['Best_Score'].mean(),4),
                       'Fit_top rate': round(g_sub['Fit_top'].mean(),2),
                       'Fit_elite rate': round(g_sub['Fit_elite'].mean(),2),
                       'n_constraints': 0})

for r in range(1, 5):
    for combo in _comb(ATTRS, r):
        mask = np.ones(len(df), dtype=bool)
        for c in combo: mask &= (df[c].values==1)
        sub = df[mask]
        if len(sub) >= 2:
            combo_rows.append({'Combination':'+'.join(combo), 'n':len(sub),
                               'Mean score': round(sub['Best_Score'].mean(),4),
                               'Fit_top rate': round(sub['Fit_top'].mean(),2),
                               'Fit_elite rate': round(sub['Fit_elite'].mean(),2),
                               'n_constraints': r})

combo_df = pd.DataFrame(combo_rows).sort_values('Mean score', ascending=False)
top15    = combo_df.head(15)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax = axes[0]
score_range = top15['Mean score'].max()-top15['Mean score'].min()+1e-9
colors_bar  = plt.cm.RdYlGn(
    (top15['Mean score'].values-top15['Mean score'].min())/score_range)
ax.barh(top15['Combination'], top15['Mean score'],
        color=colors_bar, alpha=0.85, edgecolor='white')
ax.axvline(13.75, color='#e67e22', linewidth=1.2, linestyle='--', label='Fit_top (13.75)')
ax.axvline(14.5,  color='#c0392b', linewidth=1.2, linestyle=':',  label='Fit_elite (14.5)')
ax.set_xlabel('Mean Best_Score', fontsize=10)
ax.set_title('Top 15 combinations by mean score', fontsize=10)
ax.invert_yaxis(); ax.legend(fontsize=8)
for i,(score,n) in enumerate(zip(top15['Mean score'], top15['n'])):
    ax.text(score+0.01, i, f'{score:.3f}  (n={n})', va='center', fontsize=7.5)

ax2 = axes[1]
elite_rates = top15['Fit_elite rate'].values
cols2 = ['#c0392b' if v>=0.6 else '#e67e22' if v>=0.3 else '#95a5a6'
         for v in elite_rates]
ax2.barh(top15['Combination'], elite_rates, color=cols2, alpha=0.85, edgecolor='white')
ax2.set_xlabel('Fit_elite rate  (score ≥ 14.5)', fontsize=10)
ax2.set_title('Fit_elite rate by combination', fontsize=10)
ax2.invert_yaxis(); ax2.set_xlim(0, 1.15)
for i,v in enumerate(elite_rates):
    ax2.text(v+0.01, i, f'{v:.2f}', va='center', fontsize=7.5)

plt.suptitle('Step 4.3 — Constraint combination performance  (n≥2)',
             fontsize=11, y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# ── 4.4  Coherence matrix ─────────────────────────────────────────────────────
cv_vals = {}
el = []
for ci in ATTRS:
    others = [c for c in ATTRS if c!=ci]
    for _, grp in df.groupby(others):
        if len(grp)!=2: continue
        row0=grp[grp[ci]==0].iloc[0]; row1=grp[grp[ci]==1].iloc[0]
        from_gen = sum(int(row0[c]) for c in ATTRS if c!=ci)==0
        el.append({'constraint':ci,
                   'delta':float(row1['Best_Score']-row0['Best_Score']),
                   'from_generic':from_gen})
el_df = pd.DataFrame(el)
for ci in ATTRS:
    sub = el_df[(el_df['constraint']==ci)&(~el_df['from_generic'])]['delta']
    mu=sub.mean(); sig=sub.std()
    cv_vals[ci] = round(sig/abs(mu),2) if abs(mu)>1e-9 else np.inf

snr_by_c = {c:0.0 for c in ATTRS}
for ci,cj in pairs:
    ii,jj = CONSTRAINTS.index(ci), CONSTRAINTS.index(cj)
    snr_by_c[ci] = max(snr_by_c[ci], mat_snr[ii,jj])
    snr_by_c[cj] = max(snr_by_c[cj], mat_snr[ii,jj])

fca_count = {c:0 for c in ATTRS}
for P,C in impl_top+impl_elite:
    for c in list(P)+list(C): fca_count[c]+=1

lift_by_c = dict(zip(lift_df['Constraint'], lift_df['Lift']))

coh_rows = []
for c in ATTRS:
    cv=cv_vals[c]; snr=round(snr_by_c[c],2); lft=lift_by_c.get(c,1.0); fca=fca_count[c]
    signal    = ('Strong' if (lft>1.5 or lft<0.5) and fca>0
                 else 'Moderate' if lft>1.1 or lft<0.8 else 'Weak')
    direction = ('Negative' if lft<0.7 else 'Positive' if lft>1.2 else 'Neutral/context')
    coh_rows.append({'C':c,'Name':CNAMES[c],
                     'CV (Step1)': cv if not np.isinf(cv) else 'inf',
                     'Max SNR (Step2)': snr,
                     'Lift (Step4)': round(lft,2) if not np.isinf(lft) else 'inf',
                     'FCA rules': fca, 'Signal': signal, 'Direction': direction})
coh_df = pd.DataFrame(coh_rows)

def style_signal(val):
    return {'Strong':'background-color:#d5f5e3;font-weight:bold',
            'Moderate':'background-color:#fef9e7'}.get(val,'background-color:#f9f9f9')
def style_dir(val):
    return {'Negative':'color:#c0392b;font-weight:bold',
            'Positive':'color:#27ae60;font-weight:bold'}.get(val,'color:#7f8c8d')

display(coh_df.style
    .map(style_signal, subset=['Signal'])
    .map(style_dir,    subset=['Direction'])
    .set_caption('Step 4.4 — Coherence matrix.')
    .set_table_styles([
        {'selector':'th','props':[('background-color','#2c3e50'),('color','white'),
                                   ('font-size','12px'),('padding','6px 12px')]},
        {'selector':'td','props':[('font-size','11px'),('padding','4px 12px'),
                                   ('text-align','center')]},
        {'selector':'caption','props':[('font-size','12px'),('font-weight','bold'),
                                        ('padding','6px 0'),('text-align','left')]}])
)

---
## Step 5 — Regression Model & Importance Table

**5a — LASSO feature selection**  
41 candidate terms (6 main + 15 two-way + 20 three-way interactions).
LassoCV (10-fold) with alpha range tuned for the narrow score range [12, 15].

**5b — OLS + importance table**  
Standardised OLS on LASSO-selected terms.
Continuous float scores improve regression quality vs the previous run.

In [ ]:
from itertools import combinations as _comb
from sklearn.linear_model import LassoCV, LinearRegression, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
import warnings; warnings.filterwarnings('ignore')

feat_names = list(ATTRS)
feat_cols  = {c: df[c].values.astype(float) for c in ATTRS}
for ci,cj in _comb(ATTRS,2):
    name=f'{ci}x{cj}'; feat_names.append(name)
    feat_cols[name]=feat_cols[ci]*feat_cols[cj]
for ci,cj,ck in _comb(ATTRS,3):
    name=f'{ci}x{cj}x{ck}'; feat_names.append(name)
    feat_cols[name]=feat_cols[ci]*feat_cols[cj]*feat_cols[ck]

X_full = np.column_stack([feat_cols[f] for f in feat_names])
y      = df['Best_Score'].values.astype(float)
X_std  = StandardScaler().fit_transform(X_full)

# Narrow score range [12,15] → smaller alphas needed
alphas   = np.logspace(-4, 0, 100)
lasso_cv = LassoCV(cv=10, max_iter=50000, random_state=42, alphas=alphas)
lasso_cv.fit(X_std, y)

selected = [feat_names[i] for i in range(len(feat_names))
            if abs(lasso_cv.coef_[i]) > 1e-6]
lasso_r2 = r2_score(y, lasso_cv.predict(X_std))

print(f'Feature matrix  : {X_full.shape[0]} × {X_full.shape[1]}')
print(f'Best alpha (CV) : {lasso_cv.alpha_:.5f}')
print(f'Selected terms  : {len(selected)} / {len(feat_names)}')
print(f'R² (LASSO)      : {lasso_r2:.3f}')
print('Selected:', selected)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
ax = axes[0]
ax.semilogx(lasso_cv.alphas_, lasso_cv.mse_path_.mean(axis=-1),
            color='#3498db', linewidth=1.5)
ax.axvline(lasso_cv.alpha_, color='#e74c3c', linewidth=1.8, linestyle='--',
           label=f'Best α={lasso_cv.alpha_:.5f}')
ax.set_xlabel('Alpha', fontsize=10); ax.set_ylabel('Mean CV MSE', fontsize=10)
ax.set_title('5a — LassoCV MSE path', fontsize=10)
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

ax2 = axes[1]
n_nonzero = []
for alpha in lasso_cv.alphas_:
    l = Lasso(alpha=alpha, max_iter=50000); l.fit(X_std, y)
    n_nonzero.append(np.sum(np.abs(l.coef_)>1e-6))
ax2.semilogx(lasso_cv.alphas_, n_nonzero, color='#2ecc71', linewidth=1.5)
ax2.axvline(lasso_cv.alpha_, color='#e74c3c', linewidth=1.8, linestyle='--',
            label=f'{len(selected)} terms')
ax2.set_xlabel('Alpha', fontsize=10); ax2.set_ylabel('Non-zero terms', fontsize=10)
ax2.set_title('5a — Sparsity path', fontsize=10)
ax2.legend(fontsize=9); ax2.grid(True, alpha=0.3)
plt.suptitle(f'Step 5a — LASSO path  (41 candidates → {len(selected)} selected)',
             fontsize=11, y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
selected_ols = selected if selected else list(ATTRS)

X_sel     = np.column_stack([feat_cols[f] for f in selected_ols])
X_sel_std = StandardScaler().fit_transform(X_sel)
ols       = LinearRegression().fit(X_sel_std, y)
y_pred    = ols.predict(X_sel_std)
ols_r2    = r2_score(y, y_pred)
residuals = y - y_pred

print(f'R² (OLS)      : {ols_r2:.3f}')
print(f'Residual std  : {residuals.std():.4f}  '
      f'({residuals.std()/y.mean()*100:.2f}% of mean)')

order_map = {f:1 for f in ATTRS}
for ci,cj in _comb(ATTRS,2):     order_map[f'{ci}x{cj}']=2
for ci,cj,ck in _comb(ATTRS,3):  order_map[f'{ci}x{cj}x{ck}']=3

std_coefs  = ols.coef_
abs_coefs  = np.abs(std_coefs)
total_abs  = abs_coefs.sum()
idx_sorted = np.argsort(-abs_coefs)
top4_pairs = set(summary.head(4)['Pair'].tolist())

imp_rows = []
for rank, idx in enumerate(idx_sorted, 1):
    f=selected_ols[idx]; coef=std_coefs[idx]
    snr_match = 'yes' if any(
        f'{ci}x{cj}'==f or f'{cj}x{ci}'==f
        for p in top4_pairs for ci,cj in [p.split('+')]) else ''
    imp_rows.append({'Rank':rank,'Term':f,'Order':order_map[f],
                     'Coef (std)':round(coef,4),'Abs':round(abs_coefs[idx],4),
                     'Share %':round(100*abs_coefs[idx]/total_abs,1),
                     'Direction':'+' if coef>0 else '-','SNR match':snr_match})
imp_df = pd.DataFrame(imp_rows)

def style_dir(v):
    return {'+'  :'color:#27ae60;font-weight:bold;font-size:14px',
            '-'  :'color:#e74c3c;font-weight:bold;font-size:14px'}.get(v,'')
def style_order(v):
    return {1:'background-color:#eaf4fb',2:'background-color:#eafaf1',
            3:'background-color:#fef9e7'}.get(v,'')
def style_share(v):
    try: f=float(v)
    except: return ''
    if f>=15: return 'background-color:#d5f5e3;font-weight:bold'
    if f>=8:  return 'background-color:#fef9e7'
    return ''
def style_snr(v):
    return 'background-color:#fdecea;font-weight:bold' if v=='yes' else ''

display(
    imp_df.style
    .map(style_dir,   subset=['Direction'])
    .map(style_order, subset=['Order'])
    .map(style_share, subset=['Share %'])
    .map(style_snr,   subset=['SNR match'])
    .set_caption(
        f"Step 5b — Tableau d'importance des contributions performatives  "
        f"(OLS on {len(selected_ols)} LASSO-selected terms, standardised, R²={ols_r2:.3f})")
    .bar(subset=['Share %'], color=['#e74c3c','#27ae60'],
         vmin=0, vmax=imp_df['Share %'].max(), align='zero')
    .set_table_styles([
        {'selector':'caption','props':[('font-size','12px'),('font-weight','bold'),
                                        ('padding','8px 0'),('text-align','left')]},
        {'selector':'th','props':[('background-color','#2c3e50'),('color','white'),
                                   ('font-size','12px'),('padding','6px 14px')]},
        {'selector':'td','props':[('font-size','11px'),('padding','5px 14px'),
                                   ('text-align','center')]}])
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

ax = axes[0]
order_colors = {1:'#3498db', 2:'#2ecc71', 3:'#f39c12'}
idx_by_coef  = np.argsort(std_coefs)
bar_labels   = [selected_ols[i] for i in idx_by_coef]
bar_values   = [std_coefs[i]    for i in idx_by_coef]
bar_colors   = [order_colors[order_map[selected_ols[i]]] for i in idx_by_coef]
bars = ax.barh(bar_labels, bar_values, color=bar_colors, alpha=0.85, edgecolor='white')
ax.axvline(0, color='#555', linewidth=1)
for bar, val in zip(bars, bar_values):
    ax.text(val+np.sign(val)*0.002, bar.get_y()+bar.get_height()/2,
            f'{val:+.4f}', va='center',
            ha='left' if val>=0 else 'right', fontsize=7.5)
ax.legend(handles=[
    mpatches.Patch(color='#3498db', alpha=0.85, label='Order 1 — main'),
    mpatches.Patch(color='#2ecc71', alpha=0.85, label='Order 2 — 2-way'),
    mpatches.Patch(color='#f39c12', alpha=0.85, label='Order 3 — 3-way'),
], fontsize=8, loc='lower right')
ax.set_xlabel('Standardised OLS coefficient', fontsize=10)
ax.set_title('Importance waterfall', fontsize=10, pad=8)
ax.grid(True, alpha=0.2, axis='x')

ax2 = axes[1]
sc = ax2.scatter(y_pred, y, c=residuals, cmap='RdYlGn', s=60, alpha=0.85,
                 edgecolors='white', linewidths=0.5,
                 vmin=-np.abs(residuals).max(), vmax=np.abs(residuals).max())
lims=[min(y.min(),y_pred.min())-0.1, max(y.max(),y_pred.max())+0.1]
ax2.plot(lims, lims, 'k--', linewidth=1, alpha=0.5, label='Perfect fit')
ax2.set_xlabel('Predicted', fontsize=10); ax2.set_ylabel('Actual', fontsize=10)
ax2.set_title(f'Predicted vs Actual  (R²={ols_r2:.3f})', fontsize=10)
ax2.legend(fontsize=9); ax2.grid(True, alpha=0.3)
plt.colorbar(sc, ax=ax2, label='Residual', fraction=0.046, pad=0.04)
for i in np.argsort(-np.abs(residuals))[:4]:
    ax2.annotate(df['Architecture'].iloc[i], (y_pred[i], y[i]),
                 textcoords='offset points', xytext=(6,3), fontsize=7, color='#555')
plt.suptitle(f'Step 5b — OLS  ({len(selected_ols)} terms, R²={ols_r2:.3f}, '
             f'residual std={residuals.std():.4f})', fontsize=11, y=1.02)
plt.tight_layout(); plt.show()

print()
print('Top 3 contributors:')
for row in imp_rows[:3]:
    print(f'  [{row["Rank"]}] {row["Term"]:<22}  {row["Direction"]}  share={row["Share %"]}%')

---
## Step 5b — Spectral Decomposition of the Fitness Landscape (Walsh-Hadamard)

The LASSO model (Step 5a) identifies the *predictively useful* terms under sparsity
constraints. The **Walsh-Hadamard spectral decomposition** gives the *exact* decomposition
of the fitness variance over the {0,1}⁶ hypercube — no approximation, no regularisation.

### Theoretical background

Any function f : {0,1}⁶ → ℝ decomposes uniquely as:

$$f(x) = \sum_{S \subseteq \{S,A,R,I,H,L\}} \hat{f}(S) \cdot \chi_S(x)$$

where $\chi_S(x) = (-1)^{\sum_{i \in S} x_i}$ are the Walsh basis functions and:

$$\hat{f}(S) = \frac{1}{2^6} \sum_{x \in \{0,1\}^6} f(x) \cdot \chi_S(x)$$

**Parseval's identity** gives the exact variance decomposition:

$$\text{Var}(f) = \sum_{S \neq \emptyset} \hat{f}(S)^2$$

Each $\hat{f}(S)^2$ is the **exact contribution of interaction S to the total variance** —
no collinearity, no sampling noise (all 64 points observed exactly once).

| Quantity | Meaning |
|---|---|
| $\hat{f}(\emptyset)$ | Mean fitness |
| $\hat{f}(\{R\})$ | Main effect of R |
| $\hat{f}(\{R,H\})$ | Pure R×H epistatic interaction |
| $\sum_{|S|=1} \hat{f}(S)^2 / \text{Var}(f)$ | Fraction of variance due to main effects |
| $\sum_{|S|=2} \hat{f}(S)^2 / \text{Var}(f)$ | Fraction due to pairwise interactions |

> Sign convention: $\chi_S(x) = (-1)^{\sum_{i\in S} x_i}$, so $\hat{f}(\{R\}) < 0$
> means activating R (x_R = 1) *increases* fitness.

In [ ]:
# =============================================================================
# Step 5b.1 — Walsh-Hadamard Transform
# =============================================================================
from itertools import combinations as _comb

CONSTRAINTS = ['S', 'A', 'R', 'I', 'H', 'L']
n_vars = len(CONSTRAINTS)
N = 2**n_vars  # 64

# ── Build ordered fitness vector ──────────────────────────────────────────────
# Index i: bit j of i = value of CONSTRAINTS[j]
y_vec = np.zeros(N)
for i in range(N):
    bits = [(i >> j) & 1 for j in range(n_vars)]
    mask = np.ones(len(df), dtype=bool)
    for j, c in enumerate(CONSTRAINTS):
        mask &= (df[c].values == bits[j])
    assert mask.sum() == 1, f'Missing architecture for bits={bits}'
    y_vec[i] = df[mask]['Best_Score'].values[0]

# ── Walsh-Hadamard Transform (fast butterfly) ─────────────────────────────────
def wht(f):
    """Normalised WHT: returns f_hat(S) for all 2^n subsets S."""
    result = f.copy().astype(float)
    k = int(np.log2(len(result)))
    for i in range(k):
        step = 2**(i+1)
        for j in range(0, len(result), step):
            for l in range(j, j + step//2):
                a, b = result[l], result[l + step//2]
                result[l]           = a + b
                result[l + step//2] = a - b
    return result / N

f_hat = wht(y_vec)

# ── Subset labelling ──────────────────────────────────────────────────────────
def idx_to_subset(s):
    return frozenset(CONSTRAINTS[j] for j in range(n_vars) if (s >> j) & 1)

total_var = float(np.sum(f_hat[1:]**2))   # Parseval identity

print(f'f_hat(∅)   = {f_hat[0]:.4f}  (mean fitness)')
print(f'Var(f)     = {total_var:.6f}  (Parseval)')
print(f'Std(f)     = {np.sqrt(total_var):.4f}  (check: sample std = {y_vec.std():.4f})')
print()

# ── Coefficient table ─────────────────────────────────────────────────────────
coef_rows = []
for s in range(1, N):
    sub = idx_to_subset(s)
    coef_rows.append({
        'Subset'     : '+'.join(sorted(sub)),
        'Order'      : len(sub),
        'f_hat'      : round(f_hat[s], 6),
        'f_hat2'     : round(f_hat[s]**2, 6),
        'Var share %': round(100 * f_hat[s]**2 / total_var, 3),
    })
coef_df = pd.DataFrame(coef_rows).sort_values('f_hat2', ascending=False)

# ── Variance by order ─────────────────────────────────────────────────────────
print('Variance decomposition by interaction order:')
order_var = {}
for order in range(1, n_vars+1):
    v = coef_df[coef_df['Order']==order]['f_hat2'].sum()
    order_var[order] = v
    bar = '█' * int(v/total_var * 50)
    print(f'  Order {order}:  {v:.6f}  ({100*v/total_var:5.2f}%)  {bar}')
print()
print('Top 15 Walsh coefficients by variance share:')
print(coef_df[['Subset','Order','f_hat','f_hat2','Var share %']].head(15).to_string(index=False))

In [ ]:
# ── Artefact 1 : Variance by order ───────────────────────────────────────────
order_cols = ['#3498db','#2ecc71','#f39c12','#e74c3c','#9b59b6','#1abc9c']
orders  = list(range(1, n_vars+1))
shares  = [100*order_var[o]/total_var for o in orders]
vars_abs = [order_var[o] for o in orders]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

ax = axes[0]
bars = ax.bar(orders, shares, color=[order_cols[o-1] for o in orders],
              alpha=0.85, edgecolor='white')
for bar, s in zip(bars, shares):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.4,
            f'{s:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_xlabel('Interaction order  |S|', fontsize=10)
ax.set_ylabel('% of total variance  f̂(S)² / Var(f)', fontsize=10)
ax.set_title('Parseval variance decomposition by order\n'
             '(main effects vs epistatic interactions)', fontsize=10)
ax.set_xticks(orders)
ax.set_xticklabels([f'Order {o}\n({len(list(_comb(CONSTRAINTS,o)))} terms)' for o in orders],
                   fontsize=8)
ax.set_ylim(0, max(shares)*1.22)
ax.grid(True, alpha=0.3, axis='y')
ax.axhline(50, color='#e74c3c', linewidth=0.8, linestyle=':', alpha=0.6, label='50%')
ax.legend(fontsize=8)

ax2 = axes[1]
top20 = coef_df.head(20)
bar_colors2 = [order_cols[r-1] for r in top20['Order']]
ax2.barh(top20['Subset'], top20['f_hat'],
         color=bar_colors2, alpha=0.85, edgecolor='white')
ax2.axvline(0, color='#555', linewidth=0.9)
ax2.set_xlabel("Walsh coefficient  f̂(S)  (sign: − means activating S increases fitness)",
               fontsize=9)
ax2.set_title('Top 20 Walsh coefficients  (signed)\n'
              '(colour = interaction order)', fontsize=10)
ax2.invert_yaxis()
ax2.grid(True, alpha=0.2, axis='x')
for i, (val, name) in enumerate(zip(top20['f_hat'], top20['Subset'])):
    ax2.text(val + np.sign(val)*0.003, i, f'{val:+.4f}',
             va='center', ha='left' if val>=0 else 'right', fontsize=7.5)
ax2.legend(handles=[mpatches.Patch(color=order_cols[o-1], alpha=0.85,
           label=f'Order {o}') for o in range(1, n_vars+1)],
           fontsize=7.5, loc='lower right')

plt.suptitle(f'Step 5b — Walsh-Hadamard spectral decomposition\n'
             f'Total variance = {total_var:.4f}  |  mean fitness = {f_hat[0]:.4f}',
             fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Artefact 2 : Complete spectrum heatmap (all 63 coefficients) ──────────────
from itertools import combinations as _comb

order_cols = ['#3498db','#2ecc71','#f39c12','#e74c3c','#9b59b6','#1abc9c']
n_per_order = [len(list(_comb(CONSTRAINTS, o))) for o in range(1, n_vars+1)]

fig, axes = plt.subplots(1, n_vars, figsize=(16, 4),
    gridspec_kw={'width_ratios': n_per_order})

vmax_all = max(abs(f_hat[s]) for s in range(1, N))

for ax, order in zip(axes, range(1, n_vars+1)):
    subs  = list(_comb(range(n_vars), order))
    labels, vals = [], []
    for combo in subs:
        s = sum(1 << j for j in combo)
        labels.append('+'.join(CONSTRAINTS[j] for j in combo))
        vals.append(f_hat[s])
    vals = np.array(vals)

    im = ax.imshow(vals.reshape(1, -1), cmap='RdYlGn',
                   norm=TwoSlopeNorm(vmin=-vmax_all, vcenter=0, vmax=vmax_all),
                   aspect='auto')
    ax.set_yticks([])
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, fontsize=6.5, rotation=90)
    ax.set_title(f'Order {order}\n{100*order_var[order]/total_var:.1f}%',
                 fontsize=9, color=order_cols[order-1], fontweight='bold')
    for i, v in enumerate(vals):
        ax.text(i, 0, f'{v:+.3f}', ha='center', va='center', fontsize=6,
                color='white' if abs(v)>vmax_all*0.5 else 'black')
    ax.spines['bottom'].set_color(order_cols[order-1])
    ax.spines['bottom'].set_linewidth(2.5)

plt.colorbar(im, ax=axes[-1], fraction=0.3, pad=0.04, label='f̂(S)')
plt.suptitle('Step 5b — Complete Walsh spectrum  (all 63 non-trivial subsets)\n'
             '(green=positive f̂, red=negative f̂  |  bottom bar colour = interaction order)',
             fontsize=11, y=1.04)
plt.tight_layout()
plt.show()

In [ ]:
# ── Artefact 3 : LASSO vs Walsh cross-reference ──────────────────────────────
print("=" * 72)
print("Step 5b — LASSO vs Walsh spectrum cross-reference")
print("=" * 72)
print(f"{'Rank':<5} {'Subset (Walsh)':<22} {'Ord':>4} {'f̂(S)':>10} "
      f"{'Var%':>7} {'LASSO coef':>12}  In LASSO")
print("-" * 72)

lasso_dict = {}
for f_name, coef in zip(selected_ols, ols.coef_):
    lasso_dict[f_name] = coef

for rank, (_, row) in enumerate(coef_df.head(20).iterrows(), 1):
    sub_str = row['Subset']
    # Try all permutations of the subset name for LASSO lookup
    parts   = sub_str.split('+')
    lasso_coef = None
    for perm in [parts, list(reversed(parts))]:
        key = 'x'.join(perm)
        if key in lasso_dict:
            lasso_coef = lasso_dict[key]
            break
    in_lasso = '✓' if lasso_coef is not None else '—'
    lc_str   = f'{lasso_coef:+.4f}' if lasso_coef is not None else '—'
    print(f"{rank:<5} {sub_str:<22} {row['Order']:>4} {row['f_hat']:>+10.4f} "
          f"{row['Var share %']:>6.2f}% {lc_str:>12}  {in_lasso}")

print()
# Summary statistics
top10_subs  = set(coef_df.head(10)['Subset'].tolist())
lasso_set   = set(selected_ols)
captured    = sum(1 for s in top10_subs
                  if s.replace('+','x') in lasso_set
                  or 'x'.join(reversed(s.split('+'))) in lasso_set)
print(f"Walsh top-10 captured by LASSO  : {captured} / 10")
print(f"Total LASSO selected terms      : {len(selected_ols)}")
print(f"LASSO R²                        : {lasso_r2:.3f}")
print(f"Walsh order-1 variance share    : {100*order_var[1]/total_var:.1f}%")
print(f"Walsh cumulative (orders 1-2)   : {100*(order_var[1]+order_var[2])/total_var:.1f}%")
print()
print("Spectral complexity indicators:")
print(f"  Effective dimensionality       : "
      f"{(sum(f_hat[s]**2 for s in range(1,N)))**2 / sum(f_hat[s]**4 for s in range(1,N)):.1f} "
      f"/ {N-1} (= Σf̂²² / Σf̂⁴  — 1 = purely additive, {N-1} = white noise)")
epi_ratio = sum(order_var[o] for o in range(2, n_vars+1)) / total_var
print(f"  Epistasis ratio                : {epi_ratio:.3f}  "
      f"({epi_ratio*100:.1f}% of variance from interactions)")